# Resolução de Conciliação de Contas

**Estratégia de Resolução:**

A abordagem ingênua seria comparar cada transação da Lista 1 com cada transação da Lista 2, resultando numa complexidade de tempo de `O(N * M)`. 

Para otimizar isso, podemos usar uma **Tabela de Dispersão** (Hash Map / Dicionário) contendo `deque`s para indexar a Lista 2. Assim, a busca por correspondências passará a ser `O(1)` para cada transação da Lista 1, totalizando uma complexidade de tempo `O(N + M)`.

### 0. Imports, Constantes e Helpers

In [3]:
import csv
from pathlib import Path
from pprint import pprint
from collections import defaultdict, deque
from datetime import date
from typing import Any, Sequence

DATE_IDX = 0
DEPARTMENT_IDX = 1
AMOUNT_IDX = 2
BENEFICIARY_IDX = 3

FOUND = "FOUND"
MISSING = "MISSING"
DATE_OFFSETS = (-1, 0, 1)

def _key(row: Sequence[Any]) -> tuple[Any, Any, Any]:
    return (
        row[DEPARTMENT_IDX],
        row[AMOUNT_IDX],
        row[BENEFICIARY_IDX],
    )

def _day(row: Sequence[Any]) -> int:
    return date.fromisoformat(str(row[DATE_IDX])).toordinal()


### 1. Carregando os Dados
Vamos iniciar carregando as duas listas de transações de arquivos CSV de exemplo.

In [5]:
transactions1 = list(csv.reader(Path('../data/reconcile_accounts/transactions1.csv').open()))
transactions2 = list(csv.reader(Path('../data/reconcile_accounts/transactions2.csv').open()))

### 2. Indexação da Lista 2 (Tabela de Dispersão)

Para otimizar as buscas, vamos agrupar (indexar) as transações da `transactions2` em um dicionário (`pool`).
- **A chave** será uma tupla contendo o departamento, o valor e o beneficiário, além da data convertida para um valor ordinal (inteiro sequencial, para facilitar as contas numéricas de `-1` e `+1` dia).
- **O valor** será um `deque` (fila eficiente) com os índices originais dessas transações na `transactions2`. Usamos uma fila pois podem haver transações exatamente idênticas (duplicadas), e cada uma deve ser correspondida ('matched') apenas uma vez, sendo removida quando lida.

In [6]:
pool = defaultdict(deque)
for i, row in enumerate(transactions2):
    pool[(_key(row), _day(row))].append(i)
pool

defaultdict(collections.deque,
            {(('Tecnologia', '16.00', 'Bitbucket'), 737763): deque([0]),
             (('Tecnologia', '49.99', 'AWS'), 737764): deque([1]),
             (('Jurídico', '60.00', 'LinkSquares'), 737763): deque([2])})

### 3. Inicialização dos Status

Começamos assumindo que nenhuma transação foi encontrada (`MISSING`), criando listas contendo o respectivo status para cada um dos itens (mesmo tamanho das listas originais de transações).

In [7]:
statuses1 = [MISSING] * len(transactions1)
statuses1

['MISSING', 'MISSING', 'MISSING']

In [8]:
statuses2 = [MISSING] * len(transactions2)
statuses2

['MISSING', 'MISSING', 'MISSING']

### 4. Processando a Lista 1 (Busca da Correspondência)

Agora iteramos sobre a `transactions1` e procuramos por ocorrências no nosso `pool` (dicionário da lista 2).

Como as datas podem diferir no máximo em 1 dia, nós buscamos a chave exata para as datas `(tx_day - 1)`, `(tx_day)` e `(tx_day + 1)`.

Se encontrarmos um 'bucket' (um `deque` de índices válido no dicionário), sabemos que houve um *match*.
Marcamos o status de ambos como `FOUND` e removemos (com `popleft()`) o respectivo índice da Lista 2 no `pool` para garantir que ele não seja usado numa próxima iteração (evitando reuso indevido no caso de duplicatas).

In [9]:
for i, row in enumerate(transactions1):
    key = _key(row)
    tx_day = _day(row)

    for offset in DATE_OFFSETS:
        bucket = pool.get((key, tx_day + offset))
        if bucket:
            statuses1[i] = FOUND
            statuses2[bucket.popleft()] = FOUND
            break


### 5. Resultados Finais

Ao final, simplesmente concatenamos os valores originais das transações com a coluna de status que foi construída na etapa anterior.

In [10]:
out1 = [list(row) + [status] for row, status in zip(transactions1, statuses1)]
out1

[['2020-12-04', 'Tecnologia', '16.00', 'Bitbucket', 'FOUND'],
 ['2020-12-04', 'Jurídico', '60.00', 'LinkSquares', 'FOUND'],
 ['2020-12-05', 'Tecnologia', '50.00', 'AWS', 'MISSING']]

In [11]:
out2 = [list(row) + [status] for row, status in zip(transactions2, statuses2)]
out2

[['2020-12-04', 'Tecnologia', '16.00', 'Bitbucket', 'FOUND'],
 ['2020-12-05', 'Tecnologia', '49.99', 'AWS', 'MISSING'],
 ['2020-12-04', 'Jurídico', '60.00', 'LinkSquares', 'FOUND']]